# Banking Customer Analytics & Risk Engine — Exploratory Data Analysis (EDA)

This notebook conducts exploratory analysis and hypothesis testing on the loan application dataset, answering Analytical Questions **AQ-01** through **AQ-08**.

All business and data logic is imported from `banking_analytics` to ensure reproducibility.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from banking_analytics.utils.config_loader import load_config
from banking_analytics.utils.io_utils import read_csv_safe
from banking_analytics.eda import (
    answer_aq01_overall_approval,
    answer_aq02_credit_score_analysis,
    answer_aq03_income_analysis,
    answer_aq04_dti_analysis,
    answer_aq05_loan_purpose_analysis,
    answer_aq06_employment_analysis,
    answer_aq07_guarantees_analysis,
    answer_aq08_demographic_equity,
    run_full_eda
)
from banking_analytics.db_loader import DatabaseManager

config = load_config("../config/config.yaml")
df = read_csv_safe(f"../{config['data']['processed_path']}")
print(f"Loaded clean dataset: {df.shape[0]} rows, {df.shape[1]} columns")

## 1. Overall Approval Rate (AQ-01)

In [ ]:
aq01 = answer_aq01_overall_approval(df)
print(f"Total applicants: {aq01['total_applicants']}")
print(f"Approval Rate: {aq01['approval_pct']}%")
print(f"Rejection Rate: {aq01['rejection_pct']}%")

## 2. Credit Score Analysis & Mann-Whitney U Test (AQ-02)

In [ ]:
aq02 = answer_aq02_credit_score_analysis(df)
pd.DataFrame(aq02['bands'])

## 3. Debt-to-Income (DTI) and Annual Income (AQ-03 & AQ-04)

In [ ]:
aq03 = answer_aq03_income_analysis(df)
pd.DataFrame(aq03['quartiles'])

In [ ]:
aq04 = answer_aq04_dti_analysis(df)
pd.DataFrame(aq04['bands'])

## 4. SQL Analytics via DuckDB

In [ ]:
db = DatabaseManager(config)
db.load_data(df)
db.create_views("../sql/queries/views")

print("Credit risk summary from SQL view:")
df_view = db.run_query("SELECT * FROM v_credit_risk_summary")
display(df_view)
db.close()